# Ashvini: train every model on Kaggle

Runs on a free Kaggle notebook (GPU T4 x2 or P100). Add the datasets you have as Kaggle inputs, edit the paths in the config cell, and run all cells. Modules without data are skipped. Results land in `/kaggle/working/models`: a model card (`.json` and `.md`) per module, the serving model `collision-risk.onnx`, and `index.json`.

Data sources (download them yourself; check each licence):
- ESA Collision Avoidance Challenge: Zenodo record 4463683
- NASA OMNI hourly: omniweb.gsfc.nasa.gov (omni2_all_years.dat)
- ESA-ADB: the benchmark's own repository, exported to a parquet of channels plus a labels CSV
- SPEED+: Stanford Digital Repository
- Space-Track element history and CelesTrak GP: for orbit error, manoeuvres and debris. Space-Track data must not be published.

In [ ]:
!git clone -q https://github.com/Mahendr99ar/ashvini.git
%cd ashvini/ml
!pip -q install -e ".[rl]"

In [ ]:
# 1. A quick synthetic run proves the setup works (about a minute)
!python -m ashvini_ml.train_all --synthetic --quick --out /tmp/smoke

In [ ]:
# 2. Point at your data. Delete lines for modules you have no data for.
import json
config = {
    "collision": {"data": "/kaggle/input/esa-collision-avoidance/train_data.csv"},
    "kp": {"omni": ["/kaggle/input/omni-hourly/omni2_all_years.dat"]},
    "rl": {"cdms": "/kaggle/input/esa-collision-avoidance/train_data.csv", "steps": 200000},
    # "telemetry": {"channels": "/kaggle/input/esa-adb/mission1.parquet", "labels": "/kaggle/input/esa-adb/labels.csv"},
    # "density": {"densities": "/kaggle/input/champ-density/champ.csv", "omni": ["/kaggle/input/omni-hourly/omni2_all_years.dat"]},
    # "pose": {"root": "/kaggle/input/speedplus", "epochs": 30},
}
json.dump(config, open("/kaggle/working/data.json", "w"), indent=2)

In [ ]:
!python -m ashvini_ml.train_all --config /kaggle/working/data.json --out /kaggle/working/models

In [ ]:
# 3. Read the cards. Every one says whether the model beat its baseline.
from pathlib import Path
for card in sorted(Path("/kaggle/working/models").glob("*.md")):
    print(card.read_text()[:1500], "\n" + "-" * 60)

## Next

Download `collision-risk.onnx` and `collision-risk.serving.json` into `backend/models/`, set `ASHVINI_RISK_MODEL_PATH=models/collision-risk.onnx`, and the API serves the model. Copy the other cards into the same folder and the website's Models page shows them.